# 49 · A whole worked example, MERS-CoV in South Korea, 2015, checked against pandas

## What this checks

This check covers a whole worked example, not one widget. The MERS-CoV example runs two workflows on the 2015
outbreak in South Korea. They chain these widgets:

- **Table input** reads the line list and the contacts.
- **Apply ruleset**, with the 7-1-7 ruleset release 2.0.0, gives each case three verdicts. 7-1-7 is a set of
  timeliness targets for detecting an outbreak, notifying it and responding to it.
- **Total within groups** counts the verdicts, the sources and the settings of exposure.
- **Relate**, one row per match, joins each case to the cases named as its source.
- The **transmission tree** puts each case in a chain of who infected whom.

The result is recorded against Apply ruleset release 0.17.0.

Why the whole chain? A warning once said "189 records the rules could not read", out of 162 cases. No check of one
widget on its own data could have caught that. The number came from the chain.

## How it is checked

We recompute every count and value from the example's two source files, using **pandas**, Python's library for
tables. pandas reads the files itself. Days are counted between calendar dates. The tree is built by following each
case's first listed source. Nothing here reads Fieldwork's tables.

**The data are not in this lab.** The source, the dataset `mers_korea_2015` from the R package outbreaks (Cauchemez and
others), states no licence. So it is used only in its own example repository. The lab keeps Fieldwork's results, the commit
the example pins, and a digest of each file. A digest is a fingerprint of a file's exact contents.

The check reads each file at that commit, from the example checked out beside the lab, or from GitHub. It checks
each file's digest before it reads a row.

Values are read as Fieldwork says it reads a table. Spaces around a key value are removed. Each value changed that
way is counted as the data issue *Spaces trimmed*.

## What should happen

Every value is a count or a whole number of days, so every criterion is exact.

1. **Each case.** The days from onset to diagnosis and from diagnosis to report, from the columns `dt_onset`,
   `dt_diag` and `dt_report`. Its three verdicts against the
   declared targets:
   - **Met** at or under the target.
   - **Not met** beyond it.
   - **No verdict** where a date is missing.

   Response has no date in the data, so every case has No verdict for response.
2. **Each count.** Cases by detection verdict and by notification verdict. Links by source and by setting of
   exposure.
3. **Relate, one row per match.** The same case and source rows, with "none recorded" for a case no link names. The
   same numbers related, kept, and matching several rows.
4. **The tree.** The same counts of cases, chains, links, cases naming no infector, undated cases, cases with several
   sources and further links. The same size and number of generations for each chain.
5. **The data issues.** Each counted issue's records, recomputed from the file. They are an empty onset date, a
   response date no record carries, and a value written with spaces around it.

These criteria were fixed before the check was run.

In [ ]:
NEEDED = ["pandas"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested:", NEEDED)
except ImportError:
    print("Not Pyodide; using the installed packages.")

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_49_mers import run

report = run(Path.cwd() / "fixtures" / "mers-korea-001.json")
print("agrees:", report["agrees"])
print("checked:", report["checked"])
print("source:", report["source"]["commit"][:7], {p.split('/')[-1]: d[:12] for p, d in report["source"]["files"].items()})
print("differs:", report["differs"])
print("observed:", report["observed"])

## The result on record

First recorded on 12 November 2026, against Apply ruleset release 0.13.1. Rerun on 26 November 2026, against Apply
ruleset release 0.17.0: **Agrees**.

Both workflows were recomputed from the files at the example's commit bc2d8f3. Every value matched exactly:

- 162 cases, each with its days and three verdicts;
- four count tables;
- Relate's 165 rows;
- the tree's 8 counts and its 67 chains;
- 7 data issues.

The result agrees in CPython.

The check found two defects on the way, both since fixed:

- **Missing counts.** The run's record left out Relate's counts of cases matching several rows.
- **A silent trim.** Fieldwork removed the space from one key value, "Family member ", without saying so. It now
  trims and counts each value it changes.

Running the cell above repeats the check on the same data.

## What this does not show

- That `dt_onset`, `dt_diag` and `dt_report` mean what 7-1-7 means by emergence, detection and notification. The
  practitioner chose that match, and both sides read the same columns.
- That the contacts are the true transmission, or that a case's first listed source is its infector.
- That a report dated before its diagnosis is a fact, not an error in the source. There are 7 such cases here. Both
  sides count each as a negative number of days, and so as Met.

In [ ]:
for item in report["notEstablished"]:
    print("-", item)
e = report["environment"]
print("pandas:", e["pandas"], "·", e["runtime"], "· Pyodide" if e["pyodide"] else "")